In [3]:
import pandas as pd
import numpy as np

first_invocation_bursty = 33.14

experiments = [
    {"id": "exp_20250625_014444", "name": "Openwhisk"},
    {"id": "exp_20250625_071954", "name": "NMIG"},
    {"id": "exp_20250624_210136", "name": "Histogram"},
    {"id": "exp_20250624_165438", "name": "Pagurus"},
]




COST_PER_MB_PER_SECOND = 0.00001
T_TARGET = 4 * 3600

summary = {}

for exp in experiments:
    fn = pd.read_csv(f"../results/{exp['id']}/results_updated.csv")
    df = pd.read_csv(f"../results/{exp['id']}/gpu_usage.csv")
    first_fn_timestamp_ms = fn['timestamp'].iloc[0]
    initial_time_ms = first_fn_timestamp_ms + first_invocation_bursty * 1000
    df['second_level_datetime'] = pd.to_datetime(df['Timestamp'], unit='ms').dt.floor('s')

    # avg resident GPU memory per container per second, then summed across containers
    df_avg = (df.groupby(['ContainerID', 'second_level_datetime'])['GPU_Memory_MB']
                .mean().reset_index())
    mem_per_second = (df_avg.groupby('second_level_datetime')['GPU_Memory_MB']
                            .sum().reset_index(name='total_mem_mb'))

    initial_dt = pd.to_datetime(initial_time_ms, unit='ms')
    mem_per_second['rel_s'] = ((mem_per_second['second_level_datetime'] - initial_dt)
                               .dt.total_seconds().astype(int))
    mem_per_second = mem_per_second[mem_per_second['rel_s'] >= 0].reset_index(drop=True)

    # fill every second to T_TARGET so all runs share one horizon
    full = pd.DataFrame({'rel_s': range(T_TARGET + 1)})
    m = full.merge(mem_per_second[['rel_s', 'total_mem_mb']], on='rel_s', how='left')
    m['total_mem_mb'] = m['total_mem_mb'].fillna(0)

    # --- the three metrics, all from the same series ---
    gpu_mem_seconds_mb_s = m['total_mem_mb'].sum()          # MB·s
    gpu_mem_seconds_gb_s = gpu_mem_seconds_mb_s / 1024.0    # GB·s  (resource metric)
    total_cost = gpu_mem_seconds_mb_s * COST_PER_MB_PER_SECOND  # $ (your existing metric)
    mean_resident_mb = m['total_mem_mb'].mean()             # avg held memory
    peak_resident_mb = m['total_mem_mb'].max()              # peak held memory
    # idle-memory share: fraction of GPU-mem-seconds while no inference is active
    # (only meaningful if you can mark active seconds; see note below)

    summary[exp['name']] = {
        'GPU-mem-seconds (GB·s)': gpu_mem_seconds_gb_s,
        'GPU cost ($)': total_cost,
        'Mean resident GPU mem (MB)': mean_resident_mb,
        'Peak resident GPU mem (MB)': peak_resident_mb,
    }

summary_df = pd.DataFrame(summary).T
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print(summary_df, '\n')

# reduction of NMIG vs each baseline, per metric — paste these into the paper
nmig = summary['NMIG']
for metric in summary_df.columns:
    print(f'\n{metric}:')
    for name in summary:
        if name == 'NMIG':
            continue
        base = summary[name][metric]
        red = (base - nmig[metric]) / base * 100 if base else float('nan')
        print(f'  vs {name:10s}: {red:6.2f}% reduction  (NMIG {nmig[metric]:,.3f} vs {base:,.3f})')

           GPU-mem-seconds (GB·s)  GPU cost ($)  Mean resident GPU mem (MB)  \
Openwhisk              98,783.307     1,011.541                   7,023.615   
NMIG                      506.433         5.186                      36.008   
Histogram              95,420.850       977.110                   6,784.540   
Pagurus               101,001.715     1,034.258                   7,181.347   

           Peak resident GPU mem (MB)  
Openwhisk                  22,910.000  
NMIG                       11,071.000  
Histogram                  22,928.000  
Pagurus                    22,884.000   


GPU-mem-seconds (GB·s):
  vs Openwhisk :  99.49% reduction  (NMIG 506.433 vs 98,783.307)
  vs Histogram :  99.47% reduction  (NMIG 506.433 vs 95,420.850)
  vs Pagurus   :  99.50% reduction  (NMIG 506.433 vs 101,001.715)

GPU cost ($):
  vs Openwhisk :  99.49% reduction  (NMIG 5.186 vs 1,011.541)
  vs Histogram :  99.47% reduction  (NMIG 5.186 vs 977.110)
  vs Pagurus   :  99.50% reduction  (NMIG 5.

In [4]:
summary_df

,GPU-mem-seconds (GB·s),GPU cost ($),Mean resident GPU mem (MB),Peak resident GPU mem (MB)
Openwhisk,"98,783.307","1,011.541","7,023.615","22,910.000"
NMIG,506.433,5.186,36.008,"11,071.000"
Histogram,"95,420.850",977.110,"6,784.540","22,928.000"
Pagurus,"101,001.715","1,034.258","7,181.347","22,884.000"
